<a href="https://colab.research.google.com/github/sithmi4/Statistical-Learning-e23207/blob/main/E23207_Bayesian_Inference_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment: Deep Dive into Bayesian Estimation and Gaussian Mixture Models

This comprehensive Jupyter Notebook contains thorough analytical solutions, clear step-by-step mathematical proofs, and complete interactive implementations via Plotly for four main core problems:
1. **Bayesian Estimation of a User Ability Parameter from Item Responses (IRT)**
2. **Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates**
3. **Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates**
4. **Gaussian Mixture Clustering as Conditional Updating (Derivations & EM Implementation)**


---
# Q1. Bayesian Estimation of a User Ability Parameter from Item Responses

### Task 1: Visualizing the Mechanics
The 2PL Item Response Theory (IRT) model defines the probability of a correct response as:
$$P(Y_i=1\mid \Theta=\theta)=p_i(\theta)=\frac{1}{1+e^{-a_i(\theta-b_i)}}$$

**Interpretation of shifts:**
* The difficulty parameter $b_i$ governs the horizontal positioning of the curve along the ability axis $\theta$. Specifically, $b_i$ represents the location where the probability of answering correctly is exactly $0.5$.
* As $b_i$ increases (e.g., moving from $b_i = -1$ to $b_i = 1$), the curve shifts horizontally to the **right**. This signifies a more difficult item, requiring a higher user ability $\theta$ to achieve the same probability of success.
* The discrimination parameter $a_i$ determines the slope or steepness of the curve at $\theta = b_i$. A higher value of $a_i$ results in a sharper, steeper transition, meaning the item can better distinguish between abilities slightly below and slightly above $b_i$.


In [1]:
import numpy as np
import plotly.graph_objects as go

# Generate theta grid
theta = np.linspace(-4, 4, 300)

def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

fig1 = go.Figure()

configs = [
    {'a': 2.0, 'b': -1.0, 'name': 'a=2.0, b=-1.0 (Easy, High Discrim)'},
    {'a': 2.0, 'b': 0.0,  'name': 'a=2.0, b=0.0 (Medium, High Discrim)'},
    {'a': 2.0, 'b': 1.0,  'name': 'a=2.0, b=1.0 (Hard, High Discrim)'},
    {'a': 0.6, 'b': 0.0,  'name': 'a=0.6, b=0.0 (Medium, Low Discrim)'}
]

for cfg in configs:
    fig1.add_trace(go.Scatter(
        x=theta, y=p_i(theta, cfg['a'], cfg['b']),
        mode='lines', name=cfg['name']
    ))

fig1.update_layout(
    title='2PL IRT Item Characteristic Curves (ICC)',
    xaxis_title='Latent User Ability (θ)',
    yaxis_title='Probability of Correct Response P(Y=1|θ)',
    template='plotly_white'
)
fig1.show()


### Task 2: Sequential Likelihood Contribution
For a single binary response $y_k \in \{0, 1\}$ at step $k$, the likelihood contribution is:
$$L(y_k \mid \theta) = [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k}$$

Given conditional independence, the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$ is:
$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k [p_i(\theta)]^{y_i} [1 - p_i(\theta)]^{1 - y_i}$$

### Task 3: Mathematical Formulation of the Running Update
By Bayes' Theorem, the posterior density at step $k$ satisfies:
$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k} \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

### Task 4: Dynamic Shifting Mechanics
When a user correctly answers ($y_k = 1$) a highly difficult item (large positive $b_k$), the likelihood profile is heavily shifted right. Multiplying it downweights low values of $\theta$ and scales up higher ability states, forcing the new peak to step dynamically to the **right**.

### Task 5: Tracking Certainty and Sharpness
A large $a_k$ yields a steep likelihood profile that adds massive information, reducing variance and resulting in a high, sharp posterior peak. A small $a_k$ yields a flat profile, leading to wide, uncertain posteriors.

### Task 6: Numerical Implementation of a Running Grid
We use pointwise grid initialization over `np.linspace(-4, 4, 1000)`. After updating via entry-wise multiplication, the posterior array is normalized by dividing by the absolute scalar value extracted via `np.trapezoid(posterior, grid)`.


In [2]:
np.random.seed(42)
theta_true = 0.75
n_items = 20
theta_grid = np.linspace(-4, 4, 1000)

b_items = np.random.normal(0, 1, n_items)
a_items = np.random.uniform(0.5, 2.0, n_items)

prior_density = (1.0 / np.sqrt(2 * np.pi)) * np.exp(-theta_grid**2 / 2.0)
current_posterior = prior_density.copy()

bayes_estimates = [0.0]
map_estimates = [0.0]

for k in range(n_items):
    a_k = a_items[k]
    b_k = b_items[k]
    p_true = 1.0 / (1.0 + np.exp(-a_k * (theta_true - b_k)))
    y_k = 1 if np.random.uniform(0, 1) < p_true else 0

    p_grid = 1.0 / (1.0 + np.exp(-a_k * (theta_grid - b_k)))
    likelihood = p_grid if y_k == 1 else (1.0 - p_grid)

    current_posterior = current_posterior * likelihood
    area = np.trapezoid(current_posterior, theta_grid)
    current_posterior /= area

    e_mean = np.trapezoid(theta_grid * current_posterior, theta_grid)
    e_map = theta_grid[np.argmax(current_posterior)]

    bayes_estimates.append(e_mean)
    map_estimates.append(e_map)

steps = list(range(n_items + 1))
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=steps, y=bayes_estimates, mode='lines+markers', name='Posterior Mean'))
fig2.add_trace(go.Scatter(x=steps, y=map_estimates, mode='lines+markers', name='MAP Estimate'))
fig2.add_trace(go.Scatter(x=steps, y=[theta_true]*len(steps), mode='lines', name='True Ability (0.75)', line=dict(dash='dash', color='red')))
fig2.update_layout(title='Dynamic Convergence of Ability Estimators', template='plotly_white')
fig2.show()


---
# Q2. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

### Task 1 to 5: Proofs and Formulations
Given prior $\text{Beta}(\alpha_{k-1}, \beta_{k-1})$ and Bernoulli likelihood $\theta^{y_k}(1-\theta)^{1-y_k}$:
$$f(\theta \mid \mathbf{y}^{(k)}) \propto \theta^{\alpha_{k-1}+y_k-1}(1-\theta)^{\beta_{k-1}+1-y_k-1}$$
This guarantees closed-form analytical updates:
$$\alpha_k = \alpha_{k-1} + y_k \quad \text{and} \quad \beta_k = \beta_{k-1} + 1 - y_k$$
Point estimators are direct operations:
$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \frac{\alpha_k}{\alpha_k+\beta_k}, \qquad \widehat{\theta}_{\mathrm{MAP}}^{(k)} = \frac{\alpha_k-1}{\alpha_k+\beta_k-2}$$


In [3]:
np.random.seed(42)
theta_true_ctr = 0.35
n_impressions = 100

alpha_param = 1.0
beta_param = 1.0
bayes_ctr = [alpha_param / (alpha_param + beta_param)]
map_ctr = [0.5]

for k in range(1, n_impressions + 1):
    y_k = 1 if np.random.uniform(0, 1) < theta_true_ctr else 0
    alpha_param += y_k
    beta_param += (1 - y_k)

    bayes_ctr.append(alpha_param / (alpha_param + beta_param))
    if alpha_param > 1 and beta_param > 1:
        map_ctr.append((alpha_param - 1.0) / (alpha_param + beta_param - 2.0))
    else:
        map_ctr.append(0.5)

imp_steps = list(range(n_impressions + 1))
fig4 = go.Figure()
fig4.add_trace(go.Scatter(x=imp_steps, y=bayes_ctr, mode='lines', name='Posterior Mean'))
fig4.add_trace(go.Scatter(x=imp_steps, y=map_ctr, mode='lines', name='MAP Estimate'))
fig4.add_trace(go.Scatter(x=imp_steps, y=[theta_true_ctr]*len(imp_steps), mode='lines', name='True CTR', line=dict(dash='dash', color='green')))
fig4.update_layout(title='Beta-Binomial Sequential Tracking Progression', template='plotly_white')
fig4.show()


---
# Q3. Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

### Task 1 to 5: Log-Normal Physics Model
Since $y_k = \theta K_{\text{nominal}} e^{\epsilon_k}$, using change of variables yields:
$$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( -\frac{(\log y_k - \log \theta - \log K_{\text{nominal}})^2}{2\sigma^2} \right)$$
Because $\log\theta$ sits in a non-linear format inside the exponent, integration requires grid approximations via `np.trapezoid` over $\theta \in [0.01, 1.0]$.


In [4]:
from scipy.stats import beta as beta_dist
np.random.seed(42)
theta_true_shm = 0.68
K_nominal = 50.0
sigma_shm = 0.15
n_inspections = 15

grid_shm = np.linspace(0.01, 1.0, 1000)
current_pdf = beta_dist.pdf(grid_shm, 8, 1.5)

milestones = {0: current_pdf.copy()}
shm_bayes = [np.trapezoid(grid_shm * current_pdf, grid_shm)]
shm_map = [grid_shm[np.argmax(current_pdf)]]

for k in range(1, n_inspections + 1):
    y_k = theta_true_shm * K_nominal * np.exp(np.random.normal(0, sigma_shm))
    log_diff = np.log(y_k) - np.log(grid_shm) - np.log(K_nominal)
    likelihood = (1.0 / (y_k * sigma_shm * np.sqrt(2 * np.pi))) * np.exp(- (log_diff**2) / (2 * sigma_shm**2))

    current_pdf = current_pdf * likelihood
    current_pdf /= np.trapezoid(current_pdf, grid_shm)

    shm_bayes.append(np.trapezoid(grid_shm * current_pdf, grid_shm))
    shm_map.append(grid_shm[np.argmax(current_pdf)])
    if k in [1, 2, 5, 10, 15]:
        milestones[k] = current_pdf.copy()

fig6 = go.Figure()
for m_step, m_pdf in milestones.items():
    fig6.add_trace(go.Scatter(x=grid_shm, y=m_pdf, mode='lines', name=f'Step k={m_step}'))
fig6.update_layout(title='Evolution of Bounded Posterior Densities', template='plotly_white')
fig6.show()


---
# Q4. Gaussian Mixture Clustering as Conditional Updating

### Analytical Derivations (Tasks 1-9)
* **Marginal Density:** $p(x_i) = \sum_{k=1}^K \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k)$.
* **Responsibility Matrix:** $\gamma_{ik} = \frac{\phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k)}{\sum_j \phi_j \mathscr{N}(x_i \mid \mu_j, \Sigma_j)}$, mapping back directly to the soft allocation conditional expectations $\mathbb{E}[Z_i \mid X_i = x_i]$.

### Task 10: Implementation Class Workflow


In [5]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.mixture import GaussianMixture
import plotly.express as px

# Generate Synthetic CC Data representations
np.random.seed(101)
data_raw = np.vstack([
    np.random.multivariate_normal([300, 2000], [[5000, 20000], [20000, 400000]], 1000),
    np.random.multivariate_normal([2500, 8000], [[40000, -1000], [-1000, 900000]], 600),
    np.random.multivariate_normal([7000, 15000], [[900000, 5000], [5000, 2000000]], 400)
])
df_cc = pd.DataFrame(data_raw, columns=['PURCHASES', 'CREDIT_LIMIT'])

class GMMFinancialSegmenter:
    def __init__(self, n_components=3):
        self.n_components = n_components
        self.scaler = StandardScaler()
        self.gmm = GaussianMixture(n_components=n_components, random_state=42)

    def prepare_data(self, df):
        X = df[['PURCHASES', 'CREDIT_LIMIT']].values
        X_scaled = self.scaler.fit_transform(X)
        return train_test_split(X_scaled, test_size=0.2, random_state=42)

    def fit(self, X_train):
        self.gmm.fit(X_train)
        print(f"Converged: {self.gmm.converged_} inside {self.gmm.n_iter_} iterations")

    def evaluate(self, X_test):
        print(f"Avg Test Log-Likelihood: {self.gmm.score(X_test):.4f}")

    def plot_training_assignments(self, X_train):
        x_min, x_max = X_train[:, 0].min() - 0.5, X_train[:, 0].max() + 0.5
        y_min, y_max = X_train[:, 1].min() - 0.5, X_train[:, 1].max() + 0.5
        xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))
        resp = self.gmm.predict_proba(np.c_[xx.ravel(), yy.ravel()])
        max_resp = np.max(resp, axis=1).reshape(xx.shape)

        fig = go.Figure()
        fig.add_trace(go.Contour(x=np.linspace(x_min, x_max, 100), y=np.linspace(y_min, y_max, 100), z=max_resp, colorscale='Viridis'))
        fig.add_trace(go.Scatter(x=X_train[:, 0], y=X_train[:, 1], mode='markers', marker=dict(color=self.gmm.predict(X_train), colorscale='Electric')))
        fig.update_layout(title='GMM soft contours & assignments')
        fig.show()

segmenter = GMMFinancialSegmenter()
X_train, X_test = segmenter.prepare_data(df_cc)
segmenter.fit(X_train)
segmenter.evaluate(X_test)
segmenter.plot_training_assignments(X_train)


Converged: True inside 2 iterations
Avg Test Log-Likelihood: 0.6203
